# CASMI 2026 — specs_scratch R0 (library entropy + ICEBERG, own code)

**Generated by `scratch_pkg/tools/mk_notebook.py` from `scratch_pkg/drivers/r0_kaggle.py` — do not hand-edit.**
Spec: `specs/specs_scratch/`. The `casmi_scratch` package is embedded below and written to disk at start-up.

Settings: **GPU T4**, **Internet OFF**. Inputs: the competition data, **casmi26-iceberg-offline**, the COCONUT
arrays (`coco_meta.pkl`, `coco_mass.npy`), and the RDKit 2026.03.3 wheel.

R0 = the Track 3 simple-sim base (LB 0.289; 0.296 with MAX_PEAKS=100) ported with fixes: one `max_peaks` for the
library, MetFrag-lite and ICEBERG similarities; no globals; fast commit runs 20 molecules deep.

In [ ]:
CONFIG_OVERRIDES = dict(use_sim=False)          # e.g. dict(use_sim=False) for the no-simulation control
RUN_TAG = 'r0-nosim'

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
INPUT = os.environ.get("CASMI_INPUT", "/kaggle/input")    # overridable for a local dry run
WORK = Path(os.environ.get("CASMI_WORK", "/kaggle/working"))
PKG_FILES = {'casmi_scratch/__init__.py': '"""casmi_scratch: our own CASMI 2026 pipeline (specs/specs_scratch).\n\nPure-python modules (numpy / pandas / pyarrow / numba; RDKit where a module says so). The ICEBERG model runs in\nits own venv through `resources/iceberg_worker.py`; this package only talks to it through files.\n"""\n__version__ = "0.1.0"\n', 'casmi_scratch/chem.py': '"""Adduct physics and formula masses (no RDKit)."""\nimport re\n\nimport numpy as np\n\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\nELECTRON = 0.00054857990\nPROTON = MASS["H"] - ELECTRON\nH2O = 2 * MASS["H"] + MASS["O"]\nNH4 = MASS["N"] + 4 * MASS["H"]\nFORMATE = MASS["C"] + 2 * MASS["H"] + 2 * MASS["O"]\nACETATE = 2 * MASS["C"] + 4 * MASS["H"] + 2 * MASS["O"]\n\n# adduct -> (n molecules, charge, mass delta): precursor m/z = (n * M + delta) / z\nADDUCTS = {\n    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - ELECTRON), "[M+Na]+": (1, 1, MASS["Na"] - ELECTRON),\n    "[M+K]+": (1, 1, MASS["K"] - ELECTRON), "[M-H2O+H]+": (1, 1, PROTON - H2O),\n    "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O), "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -ELECTRON),\n    "[M-H2O]+": (1, 1, -ELECTRON - H2O),\n    "[M+CH3OH+H]+": (1, 1, PROTON + MASS["C"] + 4 * MASS["H"] + MASS["O"]),\n    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS["C"] + 3 * MASS["H"] + MASS["N"]),\n    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),\n    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS["Cl"] + ELECTRON), "[M]-": (1, 1, ELECTRON),\n    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS["Na"] - 2 * PROTON),\n    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS["Na"] - ELECTRON), "[2M+NH4]+": (2, 1, NH4 - ELECTRON),\n    "[2M+K]+": (2, 1, MASS["K"] - ELECTRON), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),\n    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS["Na"] - 2 * PROTON),\n    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),\n}\n\n\ndef neutral_mass(mz, adduct):\n    """Vectorised precursor m/z -> neutral mass; NaN for an unknown adduct."""\n    mz = np.asarray(mz, np.float64)\n    out = np.full(len(mz), np.nan)\n    ad = np.asarray(adduct, dtype=object)\n    for a, (n, z, d) in ADDUCTS.items():\n        m = ad == a\n        if m.any():\n            out[m] = (mz[m] * z - d) / n\n    return out\n\n\ndef precursor_of(mass, adduct):\n    """Neutral exact mass -> theoretical precursor m/z (singly charged, one molecule adducts)."""\n    n, z, d = ADDUCTS[adduct]\n    return (np.asarray(mass, np.float64) * n + d) / z\n\n\n_MONO = {"C": 12.0, "H": 1.0078250319, "N": 14.0030740052, "O": 15.9949146221, "S": 31.97207069,\n         "P": 30.97376151, "F": 18.99840320, "Cl": 34.96885271, "Br": 78.9183376, "I": 126.904468,\n         "Si": 27.9769265327, "Se": 79.9165196, "B": 11.0093055, "Na": 22.98976928, "K": 38.96370649,\n         "Fe": 55.9349375, "D": 2.0141017778, "As": 74.9215965, "Li": 7.0160034, "Sn": 119.9021947}\n_TOK = re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_mass(f):\n    """Monoisotopic mass of a molecular formula string; NaN if unparseable or an exotic element."""\n    if not isinstance(f, str):\n        return np.nan\n    s, pos = 0.0, 0\n    for m in _TOK.finditer(f):\n        if m.start() != pos or m.group(1) not in _MONO:\n            return np.nan\n        pos = m.end()\n        s += _MONO[m.group(1)] * (int(m.group(2)) if m.group(2) else 1)\n    return s if pos == len(f) else np.nan\n\n\ndef query_target(nm, ppm_center):\n    """A molecule\'s neutral masses -> centred target mass (median, corrected for the timsTOF offset)."""\n    nm = np.asarray(nm, np.float64)\n    nm = nm[np.isfinite(nm)]\n    return float(np.median(nm)) / (1.0 + ppm_center / 1e6) if len(nm) else np.nan\n', 'casmi_scratch/config.py': '"""Every setting of the pipeline in one dataclass (replaces the notebooks\' global CFG)."""\nfrom dataclasses import asdict, dataclass, replace\n\nCOVERED_ADDUCTS = ("[M+H]+", "[M+NH4]+", "[M+Na]+", "[M-H]-", "[M+CH2O2-H]-", "[M+Cl]-")   # Track 3 G2b\n\n\n@dataclass(frozen=True)\nclass Config:\n    # --- candidate window (phase1 values) ---------------------------------------------------------------\n    ppm_win: float = 8.5\n    ppm_center: float = 1.45            # timsTOF precursor reads +1.45 ppm high\n    ppm_expand: tuple = (8.5, 20.0, 45.0, 80.0)\n    ppm_fallback: float = 30.0\n    backfill_ppm: float = 120.0\n    expand_backfill: bool = True\n    topn: int = 25\n\n    # --- spectrum cleaning: ONE max_peaks for library sim, MetFrag-lite and ICEBERG sim -------------------\n    int_floor: float = 0.002\n    max_peaks: int = 100\n    mz_tol: float = 0.01\n    int_power: float = 1.0\n    ent_weight: bool = True\n    prec_margin: float = 2.0            # predicted peaks above precursor + 2 Da are dropped\n\n    # --- library / MetFrag-lite -------------------------------------------------------------------------\n    struct_mass: bool = True            # library neutral mass from the label formula\n    max_bonds: int = 60\n    frag_workers: int = 4\n\n    # --- two-tier ranking ---------------------------------------------------------------------------------\n    tau: float = 0.75\n\n    # --- ICEBERG ----------------------------------------------------------------------------------------\n    use_sim: bool = True\n    sim_tier1: bool = False\n    merge_rule: str = "max"\n    covered: tuple = COVERED_ADDUCTS\n    ce_mode: str = "ev"\n    instrument: str = "QTOF"\n    ce_default: float = 40.0\n    batch: int = 128\n    batch_retry: int = 32\n    chunk: int = 20_000\n    sparse_k: int = 100\n    rate: float = 36.6                  # requests/s on one T4 (replaced by the measured rate)\n    startup_s: float = 10.4\n    hard_budget_h: float = 7.5\n    pass2_s_per_mol: float = 1.0\n    min_ok_fast: float = 0.99\n\n    # --- run mode -----------------------------------------------------------------------------------------\n    allow_fast: bool = True\n    fast_dev_molecules: int = 20\n    public_n_mol: int = 400\n    public_n_spec: int = 1213\n    force_full: bool = False\n\n    def with_(self, **kw):\n        return replace(self, **kw)\n\n    def as_dict(self):\n        return asdict(self)\n', 'casmi_scratch/kernels.py': '"""Numba kernels: spectrum cleaning and spectral entropy similarity (phase1, verbatim logic)."""\nimport numpy as np\nfrom numba import njit, prange\n\n\n@njit(cache=True, fastmath=True)\ndef clean(mz, it, floor, topk, power, ent_weight):\n    """Drop peaks below floor * max, keep the top-k by intensity (m/z order kept), normalise to sum 1,\n    and entropy-weight low-entropy spectra."""\n    n = len(mz)\n    if n == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32)\n    mx = 0.0\n    for i in range(n):\n        if it[i] > mx:\n            mx = it[i]\n    if mx <= 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32)\n    thr = floor * mx\n    c = 0\n    for i in range(n):\n        if it[i] >= thr:\n            c += 1\n    idx = np.empty(c, np.int64)\n    j = 0\n    for i in range(n):\n        if it[i] >= thr:\n            idx[j] = i\n            j += 1\n    if c > topk:\n        v = np.empty(c, np.float32)\n        for i in range(c):\n            v[i] = it[idx[i]]\n        o = np.argsort(v)[c - topk:]\n        k2 = np.empty(topk, np.int64)\n        for i in range(topk):\n            k2[i] = idx[o[i]]\n        k2.sort()\n        idx = k2\n        c = topk\n    om = np.empty(c, np.float32)\n    oi = np.empty(c, np.float32)\n    s = 0.0\n    for i in range(c):\n        om[i] = mz[idx[i]]\n        v = it[idx[i]] ** power\n        oi[i] = v\n        s += v\n    if s > 0:\n        for i in range(c):\n            oi[i] /= s\n    if ent_weight:\n        S = 0.0\n        for i in range(c):\n            if oi[i] > 0:\n                S -= oi[i] * np.log(oi[i])\n        if S < 3.0:\n            w = 0.25 + 0.25 * S\n            s2 = 0.0\n            for i in range(c):\n                oi[i] = oi[i] ** w\n                s2 += oi[i]\n            if s2 > 0:\n                for i in range(c):\n                    oi[i] /= s2\n    return om, oi\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim(qmz, qp, cmz, cp, tol):\n    i = 0\n    j = 0\n    n = len(qmz)\n    m = len(cmz)\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0:\n            SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0:\n            SB -= cp[x] * np.log(cp[x])\n    SAB = 0.0\n    tot = 0.0\n    buf = np.empty(n + m, np.float64)\n    b = 0\n    while i < n and j < m:\n        d = qmz[i] - cmz[j]\n        if d < -tol:\n            buf[b] = qp[i]\n            i += 1\n        elif d > tol:\n            buf[b] = cp[j]\n            j += 1\n        else:\n            buf[b] = qp[i] + cp[j]\n            i += 1\n            j += 1\n        b += 1\n    while i < n:\n        buf[b] = qp[i]\n        i += 1\n        b += 1\n    while j < m:\n        buf[b] = cp[j]\n        j += 1\n        b += 1\n    for x in range(b):\n        tot += buf[x]\n    if tot <= 0:\n        return 0.0\n    for x in range(b):\n        v = buf[x] / tot\n        if v > 0:\n            SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\n    """Entropy similarity of one cleaned query against library spectra `cand` (cleaned on the fly)."""\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]\n        a = off[c]\n        b = off[c + 1]\n        if b <= a:\n            continue\n        cm, cp = clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0:\n            continue\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\n    return out\n', 'casmi_scratch/library.py': '"""Reference spectral library (train.parquet) and the Class-1 direct match `lv`."""\nimport time\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom .chem import formula_mass, neutral_mass\nfrom .kernels import search\nfrom .spectra import clean_spectrum\n\n\n@dataclass\nclass Library:\n    off: np.ndarray      # (n + 1,) peak offsets\n    mz: np.ndarray       # flat m/z\n    it: np.ndarray       # flat intensities\n    nm: np.ndarray       # neutral mass per spectrum\n    ik: np.ndarray       # inchikey14 per spectrum (object)\n    order: np.ndarray    # spectrum indices sorted by nm (finite first)\n    snm: np.ndarray      # nm[order]\n    n_ok: int            # number of finite nm\n\n    def __len__(self):\n        return len(self.off) - 1\n\n    def window(self, target, tol_da):\n        snm = self.snm[:self.n_ok]\n        lo = np.searchsorted(snm, target - tol_da, "left")\n        hi = np.searchsorted(snm, target + tol_da, "right")\n        return self.order[lo:hi]\n\n\ndef build_library(off, mz, it, prec, adduct, ik, formula=None, struct_mass=True):\n    """Library from flat arrays. With struct_mass, the label formula\'s exact mass replaces the precursor-derived\n    neutral mass wherever it parses (the base\'s TIER-1 fix)."""\n    nm = neutral_mass(prec, adduct)\n    if struct_mass and formula is not None:\n        fm = {f: formula_mass(f) for f in set(formula.tolist())}\n        nm_s = np.array([fm.get(f, np.nan) for f in formula], np.float64)\n        nm = np.where(np.isfinite(nm_s), nm_s, nm)\n    ok = np.isfinite(nm)\n    order = np.argsort(np.where(ok, nm, 1e18), kind="mergesort")\n    return Library(off=np.asarray(off, np.int64), mz=np.asarray(mz, np.float32), it=np.asarray(it, np.float32),\n                   nm=nm, ik=np.asarray(ik, dtype=object), order=order, snm=nm[order], n_ok=int(ok.sum()))\n\n\ndef load_library(path, struct_mass=True, rows=None):\n    """train.parquet -> Library. rows: optional row indices to keep (validation bans)."""\n    t0 = time.time()\n    t = pq.read_table(path, columns=["inchikey14", "adduct", "precursor_mz", "molecular_formula",\n                                     "ms2_mzs", "ms2_normalized_intensities"])\n    if rows is not None:\n        t = t.take(pa.array(np.asarray(rows, np.int64)))\n    mzc = t.column("ms2_mzs").combine_chunks()\n    itc = t.column("ms2_normalized_intensities").combine_chunks()\n    lib = build_library(\n        off=mzc.offsets.to_numpy().astype(np.int64),\n        mz=mzc.values.to_numpy(zero_copy_only=False), it=itc.values.to_numpy(zero_copy_only=False),\n        prec=t.column("precursor_mz").to_numpy(zero_copy_only=False).astype(np.float64),\n        adduct=np.asarray(t.column("adduct").cast(pa.string()).to_pylist(), dtype=object),\n        ik=np.asarray(t.column("inchikey14").cast(pa.string()).to_pylist(), dtype=object),\n        formula=np.asarray(t.column("molecular_formula").cast(pa.string()).to_pylist(), dtype=object),\n        struct_mass=struct_mass)\n    # `.values` ignores a slice offset and `.offsets` index into it, so the pair stays consistent\n    print(f"[library] {len(lib):,} spectra, {lib.n_ok:,} with a neutral mass ({time.time() - t0:.1f}s)", flush=True)\n    return lib\n\n\ndef lib_sim(lib, specs, target, cfg, exclude=None):\n    """Class 1: best entropy similarity per ik14 over library spectra within +-ppm_win of target.\n\n    specs: [(mz, it)] of one molecule. exclude: optional boolean mask over library spectra to skip.\n    """\n    cand = lib.window(target, target * cfg.ppm_win / 1e6)\n    if exclude is not None and len(cand):\n        cand = cand[~exclude[cand]]\n    if len(cand) == 0:\n        return {}\n    agg = {}\n    for mz, it in specs:\n        qm, qp = clean_spectrum(mz, it, cfg)\n        if len(qm) == 0:\n            continue\n        sc = search(qm, qp, cand, lib.off, lib.mz, lib.it, cfg.mz_tol, cfg.int_floor, cfg.max_peaks,\n                    cfg.int_power, cfg.ent_weight)\n        for c, s in zip(cand, sc):\n            k = lib.ik[c]\n            if s > agg.get(k, -1.0):\n                agg[k] = float(s)\n    return agg\n', 'casmi_scratch/metfrag.py': '"""MetFrag-lite `fr`: share of (sqrt) query intensity explained by 1-2 bond-break fragments. Needs RDKit."""\nfrom multiprocessing import Pool as MPool\n\nimport numpy as np\n\nfrom .chem import ELECTRON, MASS\nfrom .kernels import clean\n\nH_ATOM = MASS["H"]\nPROTON = H_ATOM - ELECTRON\n\n\ndef mol_graph(smi):\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog("rdApp.*")\n    m = Chem.MolFromSmiles(smi)\n    if m is None:\n        return None\n    n = m.GetNumAtoms()\n    w = np.zeros(n)\n    for a in m.GetAtoms():\n        w[a.GetIdx()] = MASS.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H_ATOM\n    if (w == 0).any():\n        return None\n    return w, [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()], n\n\n\ndef _components(n, bonds, drop):\n    adj = [[] for _ in range(n)]\n    for i, (a, b) in enumerate(bonds):\n        if i in drop:\n            continue\n        adj[a].append(b)\n        adj[b].append(a)\n    seen = np.zeros(n, bool)\n    comps = []\n    for s in range(n):\n        if seen[s]:\n            continue\n        stack, cur = [s], [s]\n        seen[s] = True\n        while stack:\n            u = stack.pop()\n            for v in adj[u]:\n                if not seen[v]:\n                    seen[v] = True\n                    stack.append(v)\n                    cur.append(v)\n        comps.append(cur)\n    return comps\n\n\ndef fragment_masses(smi, max_bonds=60, max_breaks=2):\n    g = mol_graph(smi)\n    if g is None:\n        return np.zeros(0)\n    w, bonds, n = g\n    nb = len(bonds)\n    if nb == 0 or nb > max_bonds:\n        return np.array([w.sum()])\n    out = {w.sum()}\n    for i in range(nb):\n        for c in _components(n, bonds, {i}):\n            out.add(float(w[c].sum()))\n    if max_breaks >= 2:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for c in _components(n, bonds, {i, j}):\n                    out.add(float(w[c].sum()))\n    return np.array(sorted(out))\n\n\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0:\n        return 0.0\n    ion = np.sort(np.concatenate([frag_mass + dh * H_ATOM + (PROTON if mode > 0 else -PROTON) for dh in h_shifts]))\n    w = np.sqrt(np.asarray(peak_int, float))\n    tot = w.sum()\n    if tot <= 0:\n        return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)\n\n\ndef _frag_job(args):\n    smi, max_bonds = args\n    try:\n        return fragment_masses(smi, max_bonds)\n    except Exception:\n        return np.zeros(0)\n\n\ndef frag_scores(cand_smiles, specs, mode, cfg):\n    """fr per candidate: best explained share over the molecule\'s spectra. mode > 0 positive, < 0 negative."""\n    if not len(cand_smiles):\n        return np.zeros(0, np.float32)\n    jobs = [(s, cfg.max_bonds) for s in cand_smiles]\n    if cfg.frag_workers > 1:\n        with MPool(cfg.frag_workers) as mp:\n            frags = mp.map(_frag_job, jobs, chunksize=8)\n    else:\n        frags = [_frag_job(j) for j in jobs]\n    peaks = []\n    for mz, it in specs:\n        m2, i2 = clean(np.asarray(mz, np.float32), np.asarray(it, np.float32), cfg.int_floor, cfg.max_peaks, 1.0, False)\n        peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n    out = np.zeros(len(cand_smiles), np.float32)\n    for j, f in enumerate(frags):\n        out[j] = max((explain_score(f, a, b, mode=mode, tol=cfg.mz_tol) for a, b in peaks), default=0.0)\n    return out\n', 'casmi_scratch/pipeline.py': '"""R0 pipeline: per molecule, centred window -> lv (library), fr (MetFrag-lite), sv (ICEBERG) -> two-tier ranking.\n\nPass 1 plans every deep molecule and collects simulation requests; `sim.simulate` runs them under a deadline;\npass 2 scores and fills 25 slots (stage A tight window, B wider tiers append-only, C mass-nearest top-up).\nEverything is injected (library, pool, worker, frag function), so the validation harness runs the same code.\n"""\nimport time\n\nimport numpy as np\n\nfrom .chem import neutral_mass, query_target\nfrom .library import lib_sim\nfrom .metfrag import frag_scores\nfrom .rank import Slots, rank_two_tier\nfrom .sim import build_requests, sim_priority, simulate\nfrom .spectra import ce_tuple, prep, spectrum_ces, spectrum_sv\n\n\ndef scored_range(pool, target, cfg):\n    """Contiguous pool range scored by stages A + B: +-ppm_win (+-ppm_fallback if empty), widened through\n    ppm_expand until >= topn candidates."""\n    lo, hi = pool.bounds(target, cfg.ppm_win)\n    if hi == lo:\n        lo, hi = pool.bounds(target, cfg.ppm_fallback)\n    if hi - lo < cfg.topn:\n        for ppm in cfg.ppm_expand[1:]:\n            a, b = pool.bounds(target, ppm)\n            lo, hi = min(lo, a), max(hi, b)\n            if hi - lo >= cfg.topn:\n                break\n    return lo, hi\n\n\ndef molecule_target(sub, cfg):\n    nm = neutral_mass(sub.precursor_mz.to_numpy(np.float64), sub.adduct.to_numpy(dtype=object))\n    return query_target(nm, cfg.ppm_center)\n\n\ndef plan_molecule(sub, target, lib, pool, cfg, sim_tier1=False, frag_fn=frag_scores, lib_exclude=None):\n    """Pass 1 for one molecule: lv, fr on the scored range; covered query spectra; which candidates to simulate."""\n    specs = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]\n    lib_hits = lib_sim(lib, specs, target, cfg, exclude=lib_exclude)\n    ion_mode = float(np.mean([1.0 if m == "positive" else -1.0 for m in sub.ionization_mode]))\n    lo, hi = scored_range(pool, target, cfg)\n    cand = np.arange(lo, hi)\n    lv = np.array([lib_hits.get(pool.keys[c], 0.0) for c in cand], np.float32)\n    fr = frag_fn([pool.smiles[c] for c in cand], specs, ion_mode, cfg) if len(cand) else np.zeros(0, np.float32)\n    ppm = np.abs(pool.mass[cand] / target - 1.0) * 1e6\n\n    qs, n_ce_default = [], 0\n    for r in sub.itertuples():\n        if r.adduct not in cfg.covered:\n            continue\n        ces, defaulted = spectrum_ces(ce_tuple(r.collision_energy_ev), cfg.merge_rule, cfg.ce_default)\n        n_ce_default += defaulted\n        q = prep(r.ms2_mzs, r.ms2_normalized_intensities, float(r.precursor_mz), cfg)\n        if len(q[0]):\n            qs.append((r.adduct, ces, q))\n    conds = sorted({(a, c) for a, ces, _ in qs for c in ces})\n\n    valid = np.array([k is not None and len(str(k)) >= 14 for k in pool.keys[cand]], bool)\n    to_sim = valid & ((lv < cfg.tau) | sim_tier1) & bool(cfg.use_sim)\n    return dict(target=target, lib_hits=lib_hits, ion_mode=ion_mode, specs=specs,\n                fr={int(c): float(f) for c, f in zip(cand, fr)}, qs=qs, conds=conds, n_cand=len(cand),\n                n_ce_default=n_ce_default, sim_idx=cand[to_sim],\n                prio=sim_priority(lv[to_sim], fr[to_sim], ppm[to_sim]))\n\n\ndef features(plan, idx, pool, sim, cfg, frag_fn=frag_scores):\n    """lv, sv, fr, ppm for pool indices idx (fr computed now for indices pass 1 did not score)."""\n    lv = np.array([plan["lib_hits"].get(pool.keys[c], 0.0) for c in idx], np.float32)\n    miss = [int(c) for c in idx if int(c) not in plan["fr"]]\n    if miss:\n        plan["fr"].update(zip(miss, map(float, frag_fn([pool.smiles[c] for c in miss], plan["specs"],\n                                                       plan["ion_mode"], cfg))))\n    fr = np.array([plan["fr"][int(c)] for c in idx], np.float32)\n    ppm = np.abs(pool.mass[idx] / plan["target"] - 1.0) * 1e6\n    sv = np.zeros(len(idx), np.float32)\n    for j, c in enumerate(idx):\n        k = pool.keys[c]\n        for adduct, ces, q in plan["qs"]:\n            sv[j] = max(sv[j], spectrum_sv(q, [sim.get((k, adduct, ce)) for ce in ces], cfg))\n    return lv, sv, fr, ppm\n\n\ndef rank_molecule(plan, pool, sim, cfg, frag_fn=frag_scores):\n    """Pass 2 for one deep molecule -> (Slots, diag dict, stage-A features)."""\n    target, slots = plan["target"], Slots(cfg.topn)\n    cand = pool.window(target, cfg.ppm_win)\n    if len(cand) == 0:\n        cand = pool.window(target, cfg.ppm_fallback)\n    diag, stage_a = None, None\n    if len(cand):\n        lv, sv, fr, ppm = features(plan, cand, pool, sim, cfg, frag_fn)\n        order = rank_two_tier(lv, sv, fr, ppm, cfg.tau)\n        slots.take(pool, cand[order])\n        diag = dict(n_cand=len(cand), n_covered_spec=len(plan["qs"]), n_cond=len(plan["conds"]),\n                    lv_max=float(lv.max()), sv_max=float(sv.max()), fr_max=float(fr.max()),\n                    top_is_tier1=bool(lv[order[0]] >= cfg.tau))\n        stage_a = (cand, lv, sv, fr, ppm)\n    if cfg.expand_backfill and not slots.full:\n        for tier in cfg.ppm_expand[1:]:\n            wide = pool.window(target, tier)\n            new = wide[~np.isin(wide, cand)]\n            if len(new):\n                slots.take(pool, new[rank_two_tier(*features(plan, new, pool, sim, cfg, frag_fn), cfg.tau)])\n            if slots.full:\n                break\n    if not slots.full:\n        fill_mass_nearest(slots, pool, target, cfg.backfill_ppm)\n    return slots, diag, stage_a\n\n\ndef fill_mass_nearest(slots, pool, target, ppm):\n    wide = pool.window(target, ppm)\n    if len(wide):\n        slots.take(pool, wide[np.argsort(np.abs(pool.mass[wide] - target), kind="stable")])\n\n\ndef run(mols, lib, pool, cfg, call=None, deep=None, t0=None, strict=False, frag_fn=frag_scores, lib_exclude=None,\n        log=lambda s: print(s, flush=True)):\n    """Rank every molecule.\n\n    mols: [(molecule_id, sub_df)]. deep: molecule ids that get the full engine (None = all); the rest get\n    mass-nearest +-25 ppm candidates. call: IcebergWorker (None = no simulation). lib_exclude: optional\n    {molecule_id: boolean mask over library spectra} (validation bans).\n    Returns ({molecule_id: [25 smiles]}, report dict, plans).\n    """\n    t0 = time.time() if t0 is None else t0\n    times, plans, out, diags, stage_a = {}, {}, {}, {}, {}\n    sim_tier1 = cfg.sim_tier1 or strict\n\n    t = time.time()\n    for i, (mid, sub) in enumerate(mols):\n        target = molecule_target(sub, cfg)\n        if not np.isfinite(target) or (deep is not None and mid not in deep):\n            plans[mid] = dict(target=target, deep=False)\n            continue\n        excl = None if lib_exclude is None else lib_exclude.get(mid)\n        plans[mid] = dict(plan_molecule(sub, target, lib, pool, cfg, sim_tier1, frag_fn, excl), deep=True)\n        if i % 50 == 0:\n            log(f"  pass 1: {i + 1}/{len(mols)} | {time.time() - t0:.0f}s")\n    deep_ids = [m for m, p in plans.items() if p["deep"]]\n    times["pass1_s"] = round(time.time() - t, 1)\n\n    req = build_requests(({"key": pool.keys[p["sim_idx"]], "smiles": pool.smiles[p["sim_idx"]],\n                           "mass": pool.mass[p["sim_idx"]], "prio": p["prio"], "conds": p["conds"]}\n                          for p in (plans[m] for m in deep_ids)), cfg.instrument, cfg.ce_mode)\n    log(f"[pass 1] {len(deep_ids):,} deep molecules in {times[\'pass1_s\']:.0f}s; {len(req):,} simulation requests")\n\n    t = time.time()\n    if call is not None and len(req):\n        reserve = cfg.pass2_s_per_mol * len(deep_ids) + 300.0\n        sim, sim_stats = simulate(req, call, t0 + cfg.hard_budget_h * 3600.0 - reserve, cfg, strict=strict, log=log)\n    else:\n        sim, sim_stats = {}, dict(n_requests=len(req), n_predicted=0, sim_ok_share=float("nan"), sim_errors={},\n                                  failed_rows=0, capped_rows=len(req) if call is None else 0)\n    times["sim_s"] = round(time.time() - t, 1)\n\n    t = time.time()\n    for mid, _ in mols:\n        p = plans[mid]\n        if p["deep"]:\n            slots, diag, sa = rank_molecule(p, pool, sim, cfg, frag_fn)\n            if diag is not None:\n                diags[mid], stage_a[mid] = diag, sa\n        else:\n            slots = Slots(cfg.topn)\n            if np.isfinite(p["target"]):\n                fill_mass_nearest(slots, pool, p["target"], 25.0)\n        out[mid] = slots.padded()\n    times["pass2_s"] = round(time.time() - t, 1)\n\n    report = dict(times=times, n_molecules=len(mols), n_deep=len(deep_ids),\n                  ce_defaulted=int(sum(plans[m]["n_ce_default"] for m in deep_ids)), **sim_stats,\n                  tier1_rate=float(np.mean([d["top_is_tier1"] for d in diags.values()])) if diags else float("nan"),\n                  tier2_plumbing=tier2_plumbing(stage_a, plans, cfg))\n    return out, report, dict(plans=plans, diags=diags, stage_a=stage_a, sim=sim)\n\n\ndef tier2_plumbing(stage_a, plans, cfg, n=20):\n    """Zero the tier-1 winner\'s lv (as if it had no real spectrum): does sv alone keep it in the top 25?"""\n    pos = []\n    for mid in list(stage_a)[:n]:\n        cand, lv, sv, fr, ppm = stage_a[mid]\n        top = rank_two_tier(lv, sv, fr, ppm, cfg.tau)[0]\n        if lv[top] < cfg.tau or not plans[mid]["qs"]:\n            continue\n        lv2 = lv.copy()\n        lv2[top] = 0.0\n        pos.append(int(np.flatnonzero(rank_two_tier(lv2, sv, fr, ppm, cfg.tau) == top)[0]) + 1)\n    return dict(n=len(pos), in_top25=float(np.mean(np.array(pos) <= 25)) if pos else None,\n                median_rank=float(np.median(pos)) if pos else None)\n', 'casmi_scratch/pool.py': '"""Candidate pool: structures sorted by neutral exact mass, one row per ik14."""\nimport os\nimport pickle\nimport time\nfrom multiprocessing import Pool as MPool\n\nimport numpy as np\nimport pyarrow.parquet as pq\n\n\nclass CandidatePool:\n    def __init__(self, mass, keys, smiles, flags=None):\n        o = np.argsort(np.asarray(mass, np.float64), kind="stable")\n        self.mass = np.asarray(mass, np.float64)[o]\n        self.keys = np.asarray(keys, dtype=object)[o]\n        self.smiles = np.asarray(smiles, dtype=object)[o]\n        self.flags = None if flags is None else np.asarray(flags)[o]   # provenance bits (R1)\n\n    def __len__(self):\n        return len(self.mass)\n\n    def bounds(self, target, ppm):\n        """(lo, hi) index range within +-ppm of target."""\n        return (int(np.searchsorted(self.mass, target * (1 - ppm / 1e6), "left")),\n                int(np.searchsorted(self.mass, target * (1 + ppm / 1e6), "right")))\n\n    def window(self, target, ppm):\n        return np.arange(*self.bounds(target, ppm))\n\n    def without(self, keys):\n        """A copy without the given ik14 (validation\'s Class-3 arm)."""\n        keep = ~np.isin(self.keys, np.asarray(list(keys), dtype=object))\n        return CandidatePool(self.mass[keep], self.keys[keep], self.smiles[keep],\n                             None if self.flags is None else self.flags[keep])\n\n\ndef _exact_mass(smi):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.Descriptors import ExactMolWt\n    RDLogger.DisableLog("rdApp.*")\n    m = Chem.MolFromSmiles(smi)\n    return float(ExactMolWt(m)) if m is not None else np.nan\n\n\ndef build_r0_pool(coco_dir, train_path, workers=4):\n    """COCONUT + train structures (the Track 3 base pool, without the unused fingerprints).\n\n    COCONUT rows are deduplicated on ik14 (first occurrence) with their precomputed mass; train structures not in\n    COCONUT get RDKit\'s exact mass from their SMILES; rows without a finite mass are dropped.\n    """\n    t0 = time.time()\n    cm = pickle.load(open(os.path.join(coco_dir, "coco_meta.pkl"), "rb"))\n    co_mass = np.load(os.path.join(coco_dir, "coco_mass.npy")).astype(np.float64)\n    co_k14 = np.array([str(k)[:14] for k in cm["keys"]], dtype=object)\n    _, first = np.unique(co_k14, return_index=True)\n    first = np.sort(first)\n    co_k14, co_smi, co_mass = co_k14[first], np.asarray(cm["smiles"], dtype=object)[first], co_mass[first]\n\n    tr = pq.read_table(train_path, columns=["inchikey14", "normalized_smiles"]).to_pandas()\n    tr = tr.dropna().drop_duplicates("inchikey14")\n    tr = tr[~tr.inchikey14.isin(set(co_k14))]\n    smis = list(tr.normalized_smiles)\n    if workers > 1:\n        with MPool(workers) as mp:\n            tr_mass = np.array(mp.map(_exact_mass, smis, chunksize=500), np.float64)\n    else:\n        tr_mass = np.array([_exact_mass(s) for s in smis], np.float64)\n\n    mass = np.concatenate([co_mass, tr_mass])\n    keys = np.concatenate([co_k14, tr.inchikey14.to_numpy(dtype=object)])\n    smiles = np.concatenate([co_smi, tr.normalized_smiles.to_numpy(dtype=object)])\n    good = np.isfinite(mass)\n    pool = CandidatePool(mass[good], keys[good], smiles[good])\n    print(f"[pool] COCONUT {len(co_k14):,} + train-only {len(tr):,} -> {len(pool):,} structures "\n          f"({time.time() - t0:.1f}s)", flush=True)\n    return pool\n', 'casmi_scratch/rank.py': '"""Two-tier ranking and slot filling (stage A -> B -> C), strictly deduplicated on ik14."""\nimport numpy as np\n\n\ndef rank_two_tier(lv, sv, fr, ppm, tau):\n    """Order (indices) of one candidate set.\n\n    Tier 1 = real-library match lv >= tau, by lv desc (then sv, fr, |ppm|). Tier 2 = the rest by sv desc, fr desc,\n    |ppm| asc. A missing sv / fr (NaN) counts 0; a missing ppm sorts last.\n    """\n    lv, sv, fr = (np.nan_to_num(np.asarray(x, np.float64), nan=0.0) for x in (lv, sv, fr))\n    ppm = np.nan_to_num(np.asarray(ppm, np.float64), nan=np.inf)\n    t1, t2 = np.flatnonzero(lv >= tau), np.flatnonzero(lv < tau)\n    o1 = t1[np.lexsort((ppm[t1], -fr[t1], -sv[t1], -lv[t1]))]\n    o2 = t2[np.lexsort((ppm[t2], -fr[t2], -sv[t2]))]\n    return np.concatenate([o1, o2]).astype(np.int64)\n\n\nclass Slots:\n    """Up to `topn` (ik14, smiles) picks, never two with the same ik14."""\n\n    def __init__(self, topn):\n        self.topn, self.keys, self.smiles, self.seen = topn, [], [], set()\n\n    @property\n    def full(self):\n        return len(self.smiles) >= self.topn\n\n    def add(self, key, smi):\n        if self.full or key is None:\n            return False\n        k14 = str(key)[:14]\n        if len(k14) != 14 or k14 in self.seen:\n            return False\n        self.seen.add(k14)\n        self.keys.append(k14)\n        self.smiles.append(smi)\n        return True\n\n    def take(self, pool, idx_in_order):\n        for c in idx_in_order:\n            if self.full:\n                break\n            self.add(pool.keys[c], pool.smiles[c])\n\n    def padded(self, pad="CCO"):\n        return self.smiles[:self.topn] + [pad] * max(0, self.topn - len(self.smiles))\n', 'casmi_scratch/resources/__init__.py': '', 'casmi_scratch/resources/iceberg_worker.py': '"""ICEBERG forward-simulation worker for specs_simple_ms_simulator.\n\nRuns INSIDE the isolated ms-pred environment (its own torch 2.6 / DGL / rdkit 2025.03), never in\nthe notebook\'s main interpreter -- that keeps RDKit 2026.03.3 (the metric\'s version) untouched.\nThe notebook talks to it through files only:\n\n    request tsv      : key, smiles, adduct (CASMI spelling), precursor_mz, ce, instrument\n                       (TSV, not parquet: the ms-pred environment ships no pyarrow)\n    response npz     : off (n+1,) int64, mz / it float32 flat, ok bool, err str, timing json,\n                       above_n int32 / above_share float32 / above_off float32 (per row, see below)\n\nPeaks are the top `--sparse-k` by intensity, sorted by m/z, base peak = 1.0. m/z values are the\ncharged fragment masses (ICEBERG applies the adduct shift itself).\n\nPeaks above precursor_mz + `--prec-margin` (2 Da) are dropped BEFORE the top-k, the same cleaning the\norganisers applied to the test spectra. They exist because the MassSpecGym inten checkpoints have\n`include_unshifted_mz=True`: every fragment is also emitted without the adduct shift, and for\nwater-loss adducts the unshifted root lands 17-35 Da above the precursor. What was dropped is\nrecorded per row: above_n = peak count, above_share = share of raw intensity, above_off = the\nlargest dropped m/z minus the precursor (NaN when nothing was dropped). A row left with no peaks\ngets err "empty_after_prec_cut".\n\nReused unchanged by Phase 0 (gate), Phase 2 (precompute) and Phase 4 (Kaggle on-the-fly).\n\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --inp req.tsv --out resp.npz\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --hparams-only --out hp.json\n"""\nimport argparse\nimport json\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom rdkit import Chem, RDLogger\n\nimport ms_pred.common as common\nfrom ms_pred.iceberg import gen_model, inten_model, joint_model\n\nRDLogger.DisableLog("rdApp.*")\n\n# CASMI adduct spelling -> ms-pred spelling (only where they differ)\nADDUCT_MAP = {"[M-2H2O+H]+": "[M+H-2H2O]+", "[M+CH2O2-H]-": "[M+CHO2]-"}\n\n\ndef get_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--gen", required=True)\n    p.add_argument("--inten", required=True)\n    p.add_argument("--inp")\n    p.add_argument("--out", required=True)\n    p.add_argument("--device", default="cuda")\n    p.add_argument("--batch-size", type=int, default=64)\n    p.add_argument("--max-nodes", type=int, default=100)\n    p.add_argument("--sparse-k", type=int, default=100)\n    p.add_argument("--threshold", type=float, default=0.0)\n    p.add_argument("--prec-margin", type=float, default=2.0)\n    p.add_argument("--hparams-only", action="store_true")\n    return p.parse_args()\n\n\ndef _jsonable(d):\n    out = {}\n    for k, v in dict(d).items():\n        try:\n            json.dumps(v)\n            out[k] = v\n        except TypeError:\n            out[k] = str(v)\n    return out\n\n\n# PyTorch 2.6 defaults weights_only=True, which breaks legacy Lightning checkpoints containing PosixPath/hparams.\ntry:\n    from pathlib import PosixPath, WindowsPath\n    torch.serialization.add_safe_globals([PosixPath, WindowsPath])\nexcept Exception:\n    pass\n\n_orig_torch_load = torch.load\ndef _safe_torch_load(*load_args, **load_kwargs):\n    if "weights_only" not in load_kwargs:\n        load_kwargs["weights_only"] = False\n    return _orig_torch_load(*load_args, **load_kwargs)\ntorch.load = _safe_torch_load\n\n\ndef load(args):\n    try:\n        gen = gen_model.FragGNN.load_from_checkpoint(args.gen, map_location="cpu", weights_only=False)\n    except TypeError:\n        gen = gen_model.FragGNN.load_from_checkpoint(args.gen, map_location="cpu")\n\n    try:\n        inten = inten_model.IntenGNN.load_from_checkpoint(args.inten, map_location="cpu", weights_only=False)\n    except TypeError:\n        inten = inten_model.IntenGNN.load_from_checkpoint(args.inten, map_location="cpu")\n\n    return gen, inten\n\n\ndef to_peaks(s, k, prec, margin):\n    """(n, 2) array [mz, inten] -> (mz, it, above_n, above_share, above_off).\n\n    Drops peaks above prec + margin, then keeps the top-k by intensity, sorted by m/z, base = 1.\n    """\n    s = np.asarray(s, dtype=np.float64).reshape(-1, 2)\n    s = s[np.isfinite(s).all(axis=1) & (s[:, 1] > 0)]\n    above = s[:, 0] > prec + margin\n    above_n = int(above.sum())\n    above_share = float(s[above, 1].sum() / s[:, 1].sum()) if len(s) else 0.0\n    above_off = float(s[above, 0].max() - prec) if above_n else np.nan\n    s = s[~above]\n    if len(s) == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32), above_n, above_share, above_off\n    s = s[np.argsort(s[:, 1])[::-1][:k]]\n    s = s[np.argsort(s[:, 0])]\n    return (s[:, 0].astype(np.float32), (s[:, 1] / s[:, 1].max()).astype(np.float32),\n            above_n, above_share, above_off)\n\n\ndef predict_batch(model, rows, args):\n    return model.predict_mol(\n        [r["smiles"] for r in rows],\n        collision_eng=[float(r["ce"]) for r in rows],\n        precursor_mz=[float(r["precursor_mz"]) for r in rows],\n        adduct=[r["adduct"] for r in rows],\n        instrument=[r["instrument"] for r in rows],\n        threshold=args.threshold,\n        device=args.device,\n        max_nodes=args.max_nodes,\n        adduct_shift=False,\n        canonical_root_smi=True,\n        binned_out=False,\n    )["spec"]\n\n\ndef main():\n    args = get_args()\n    t0 = time.time()\n    gen, inten = load(args)\n\n    if args.hparams_only:\n        with open(args.out, "w") as f:\n            json.dump({"gen": _jsonable(gen.hparams), "inten": _jsonable(inten.hparams),\n                       "ion2onehot_pos": common.ion2onehot_pos,\n                       "instrument2onehot_pos": common.instrument2onehot_pos}, f, indent=1)\n        return\n\n    model = joint_model.JointModel(gen_model_obj=gen, inten_model_obj=inten).to(args.device)\n    model.eval()\n    model.freeze()\n    t_load = time.time() - t0\n\n    df = pd.read_csv(args.inp, sep="\t")\n    n = len(df)\n    peaks = [None] * n\n    err = [""] * n\n    above_n = np.zeros(n, np.int32)\n    above_share = np.zeros(n, np.float32)\n    above_off = np.full(n, np.nan, np.float32)\n\n    # canonicalise exactly as predict_smis.py does; reject what ms-pred cannot take\n    rows = []\n    for i, r in enumerate(df.to_dict("records")):\n        m = Chem.MolFromSmiles(r["smiles"]) if isinstance(r["smiles"], str) else None\n        ad = ADDUCT_MAP.get(r["adduct"], r["adduct"])\n        if m is None:\n            err[i] = "rdkit_parse"\n        elif ad not in common.ion2onehot_pos:\n            err[i] = f"adduct_unsupported:{r[\'adduct\']}"\n        elif r["instrument"] not in common.instrument2onehot_pos:\n            err[i] = f"instrument_unsupported:{r[\'instrument\']}"\n        elif not np.isfinite(r["ce"]):\n            err[i] = "ce_nan"\n        else:\n            rows.append(dict(r, i=i, smiles=Chem.MolToSmiles(m), adduct=ad))\n\n    t1 = time.time()\n    with torch.no_grad():\n        for b in range(0, len(rows), args.batch_size):\n            batch = rows[b:b + args.batch_size]\n            try:\n                specs = predict_batch(model, batch, args)\n            except Exception:\n                # isolate the failing molecule(s); keep the rest of the batch\n                specs = []\n                for r in batch:\n                    try:\n                        specs.append(predict_batch(model, [r], args)[0])\n                    except Exception as e:  # noqa: BLE001\n                        specs.append(None)\n                        err[r["i"]] = f"predict:{type(e).__name__}:{str(e)[:120]}"\n            for r, s in zip(batch, specs):\n                if s is not None:\n                    i = r["i"]\n                    mz_, it_, above_n[i], above_share[i], above_off[i] = to_peaks(\n                        s.detach().cpu().numpy(), args.sparse_k, float(r["precursor_mz"]), args.prec_margin)\n                    peaks[i] = (mz_, it_)\n    if args.device.startswith("cuda"):\n        torch.cuda.synchronize()\n    t_pred = time.time() - t1\n\n    empty = (np.empty(0, np.float32), np.empty(0, np.float32))\n    ok = np.array([p is not None and len(p[0]) > 0 for p in peaks])\n    for i in range(n):\n        if peaks[i] is not None and len(peaks[i][0]) == 0 and not err[i]:\n            err[i] = "empty_after_prec_cut" if above_n[i] else "empty_spectrum"\n    peaks = [p if p is not None else empty for p in peaks]\n    lens = np.array([len(p[0]) for p in peaks], dtype=np.int64)\n    off = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    mz = np.concatenate([p[0] for p in peaks]) if n else np.empty(0, np.float32)\n    it = np.concatenate([p[1] for p in peaks]) if n else np.empty(0, np.float32)\n    timing = dict(n=n, n_predicted=int(ok.sum()), t_load=t_load, t_pred=t_pred,\n                  mol_per_s=float(len(rows) / t_pred) if t_pred > 0 else None,\n                  device=args.device, batch_size=args.batch_size, prec_margin=args.prec_margin,\n                  gpu=torch.cuda.get_device_name(0) if args.device.startswith("cuda") else "cpu")\n    np.savez(args.out, off=off, mz=mz, it=it, ok=ok, err=np.array(err, dtype="U200"),\n             above_n=above_n, above_share=above_share, above_off=above_off, timing=json.dumps(timing))\n    print(json.dumps(timing))\n\n\nif __name__ == "__main__":\n    main()\n', 'casmi_scratch/sim.py': '"""ICEBERG simulation: request building, the worker subprocess, and the deadline-watchdog scheduler.\n\nThe model runs in its own venv (torch / DGL / RDKit 2025.03) through `resources/iceberg_worker.py`; this module\nonly writes request TSVs and reads response NPZs.\n"""\nimport glob\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .chem import precursor_of\n\nWORKER_SRC = Path(__file__).resolve().parent / "resources" / "iceberg_worker.py"\nREQ_COLS = ["key", "smiles", "adduct", "precursor_mz", "ce_ev", "prio", "ce", "instrument"]\n\n\ndef to_model_ce(ce_ev, prec, mode):\n    return ce_ev if mode == "ev" else ce_ev * 500.0 / prec\n\n\ndef sim_priority(lv, fr, ppm):\n    """Per-candidate simulation priority (0 = first): lv desc, fr desc, |ppm| asc."""\n    o = np.lexsort((np.asarray(ppm, np.float64), -np.asarray(fr, np.float64), -np.asarray(lv, np.float64)))\n    r = np.empty(len(o), np.int64)\n    r[o] = np.arange(len(o))\n    return r\n\n\ndef build_requests(parts, instrument="QTOF", ce_mode="ev"):\n    """Molecule parts -> one request per distinct (key, adduct, ce_ev), sorted by priority.\n\n    Each part: dict with arrays key / smiles / mass / prio and conds = [(adduct, ce_ev)]. The precursor is the\n    candidate\'s theoretical m/z, so a request is shared by every molecule that needs it (best priority kept).\n    """\n    frames = []\n    for p in parts:\n        if not len(p["key"]) or not p["conds"]:\n            continue\n        mass = np.asarray(p["mass"], np.float64)\n        for adduct, ce in p["conds"]:\n            frames.append(pd.DataFrame({"key": np.asarray(p["key"], dtype=object),\n                                        "smiles": np.asarray(p["smiles"], dtype=object), "adduct": adduct,\n                                        "precursor_mz": precursor_of(mass, adduct), "ce_ev": float(ce),\n                                        "prio": np.asarray(p["prio"], np.int64)}))\n    if not frames:\n        return pd.DataFrame({c: [] for c in REQ_COLS})\n    req = (pd.concat(frames, ignore_index=True)\n           .sort_values(["prio", "key", "adduct", "ce_ev"], kind="stable")\n           .drop_duplicates(["key", "adduct", "ce_ev"], keep="first").reset_index(drop=True))\n    req["ce"] = to_model_ce(req.ce_ev.to_numpy(), req.precursor_mz.to_numpy(), ce_mode)\n    req["instrument"] = instrument\n    return req[REQ_COLS]\n\n\ndef rows_that_fit(remaining_s, rate, startup_s):\n    return max(0, int((remaining_s - startup_s) * rate))\n\n\ndef err_counts(err):\n    return pd.Series([str(e).split(":")[0] for e in err if e], dtype=object).value_counts().to_dict()\n\n\ndef sha1(path, chunk=1 << 20):\n    h = hashlib.sha1()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\n# --- the worker ----------------------------------------------------------------------------------------\nclass IcebergWorker:\n    """Callable(req, tag, batch_size) -> (spectra [(mz, it)], ok, err, timing, wall_s)."""\n\n    def __init__(self, py, gen, inten, env, work, device="cuda", sparse_k=100, prec_margin=2.0, batch=128):\n        self.py, self.gen, self.inten, self.env = str(py), str(gen), str(inten), env\n        self.work = Path(work)\n        self.work.mkdir(parents=True, exist_ok=True)\n        self.device, self.sparse_k, self.prec_margin, self.batch = device, sparse_k, prec_margin, batch\n        self.worker = self.work / "iceberg_worker.py"\n        shutil.copyfile(WORKER_SRC, self.worker)\n\n    def __call__(self, req, tag, batch_size=None):\n        inp, out = self.work / f"req_{tag}.tsv", self.work / f"resp_{tag}.npz"\n        req[["key", "smiles", "adduct", "precursor_mz", "ce", "instrument"]].to_csv(inp, sep="\\t", index=False)\n        cmd = [self.py, str(self.worker), "--gen", self.gen, "--inten", self.inten, "--inp", str(inp),\n               "--out", str(out), "--device", self.device, "--batch-size", str(batch_size or self.batch),\n               "--sparse-k", str(self.sparse_k), "--prec-margin", str(self.prec_margin)]\n        t = time.time()\n        r = subprocess.run(cmd, text=True, capture_output=True, env=self.env, timeout=12 * 3600)\n        if r.returncode:\n            print(r.stdout[-2000:], r.stderr[-4000:])\n            raise RuntimeError(f"worker failed on {tag}")\n        z = np.load(out)\n        # read each array once: NpzFile re-reads an array on every access (the Phase-1 Colab OOM)\n        off, mz, it = z["off"], z["mz"], z["it"]\n        assert len(off) == len(req) + 1, f"{out}: {len(off) - 1} rows, expected {len(req)}"\n        spectra = [(mz[off[i]:off[i + 1]], it[off[i]:off[i + 1]]) for i in range(len(req))]\n        ok, err, timing = z["ok"], z["err"], json.loads(str(z["timing"]))\n        for f in (inp, out):\n            f.unlink(missing_ok=True)\n        return spectra, ok, err, timing, time.time() - t\n\n\ndef setup_iceberg(input_root, work, cfg, venv="/kaggle/working/iceberg_venv"):\n    """Install the offline ICEBERG wheelhouse into its own venv, verify checkpoints, return an IcebergWorker."""\n    hits = [h for h in glob.glob(f"{input_root}/**/wheelhouse", recursive=True) if os.path.isdir(h)]\n    if not hits:\n        raise FileNotFoundError("wheelhouse not found -- attach casmi26-iceberg-offline")\n    up = os.path.dirname(sorted(hits, key=len)[0])\n    man = json.load(open(os.path.join(up, "manifest.json")))\n    assert man["ce_mode"] == cfg.ce_mode and man["instrument"] == cfg.instrument, man\n    py = f"{venv}/bin/python"\n    site = os.path.join(venv, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")\n    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}\n    env.update(PYTHONPATH=site, DGLBACKEND="pytorch", CUDA_VISIBLE_DEVICES="0")\n    if not os.path.exists(py):\n        shutil.rmtree(venv, ignore_errors=True)\n        subprocess.run([sys.executable, "-m", "venv", "--without-pip", venv], check=True)\n    os.makedirs(site, exist_ok=True)\n    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", site,\n                        *sorted(glob.glob(f"{up}/wheelhouse/*.whl"))], capture_output=True, text=True, env=env)\n    assert r.returncode == 0, r.stderr[-3000:]\n    gen, inten = os.path.join(up, "ckpt", "gen.ckpt"), os.path.join(up, "ckpt", "inten.ckpt")\n    assert sha1(gen) == man["gen_sha1"] and sha1(inten) == man["inten_sha1"], "checkpoint SHA1 mismatch"\n    device = "cuda" if shutil.which("nvidia-smi") else "cpu"\n    if device == "cpu":\n        print("[WARNING] no GPU -- ICEBERG on CPU is ~15x slower; the watchdog will cap the simulation", flush=True)\n    return IcebergWorker(py, gen, inten, env, work, device=device, sparse_k=cfg.sparse_k,\n                         prec_margin=cfg.prec_margin, batch=cfg.batch)\n\n\ndef smoke(worker, cfg):\n    """Caffeine [M+H]+ at 40 eV must predict; a 120 eV off-ladder probe is reported."""\n    m = 194.0803756\n    req = pd.DataFrame({"key": ["RYYVLZVUVIJVGH"] * 2, "smiles": ["Cn1cnc2c1c(=O)n(C)c(=O)n2C"] * 2,\n                        "adduct": "[M+H]+", "precursor_mz": precursor_of(np.array([m, m]), "[M+H]+"),\n                        "ce_ev": [40.0, 120.0]})\n    req["ce"] = to_model_ce(req.ce_ev.to_numpy(), req.precursor_mz.to_numpy(), cfg.ce_mode)\n    req["instrument"] = cfg.instrument\n    sp, ok, err, timing, _ = worker(req, "smoke")\n    assert ok[0], f"ICEBERG smoke failed: {err[0]}"\n    return dict(gpu=timing.get("gpu"), n_peaks_40=int(len(sp[0][0])),\n                probe_120=dict(ok=bool(ok[1]), n_peaks=int(len(sp[1][0])), err=str(err[1])))\n\n\n# --- the scheduler -------------------------------------------------------------------------------------\ndef simulate(req, call, deadline, cfg, strict=False, log=print):\n    """Run requests in priority-ordered chunks until done or the deadline.\n\n    call: IcebergWorker-like callable. strict: re-raise a chunk that fails twice (fast commit) instead of\n    degrading. Returns (SIM {(key, adduct, ce_ev): (mz, it, prec)}, stats dict).\n    """\n    sim, errs = {}, {}\n    failed = capped = done = ci = 0\n    rate, startup = cfg.rate, cfg.startup_s\n    n_chunks = math.ceil(len(req) / cfg.chunk) if len(req) else 0\n    while done < len(req):\n        fit = rows_that_fit(deadline - time.time(), rate, startup)\n        n = min(cfg.chunk, len(req) - done, fit)\n        if fit < min(500, len(req) - done):\n            capped = len(req) - done\n            log(f"[watchdog] deadline: {capped:,} lowest-priority requests left unsimulated")\n            break\n        part = req.iloc[done:done + n]\n        try:\n            spectra, ok, err, timing, wall = call(part, f"c{ci}")\n        except RuntimeError:\n            try:\n                spectra, ok, err, timing, wall = call(part, f"c{ci}r", cfg.batch_retry)\n            except RuntimeError:\n                if strict:\n                    raise\n                failed += n\n                log(f"[WARNING] chunk {ci} ({n:,} rows) failed twice -- those candidates rank by fr/ppm")\n                done, ci = done + n, ci + 1\n                continue\n        for row, s, o in zip(part.itertuples(), spectra, ok):\n            if o:\n                sim[(row.key, row.adduct, row.ce_ev)] = (s[0], s[1], row.precursor_mz)\n        for k, v in err_counts(err).items():\n            errs[k] = errs.get(k, 0) + v\n        if timing.get("mol_per_s"):\n            rate, startup = float(timing["mol_per_s"]), max(float(wall - timing["t_pred"]), 1.0)\n        done, ci = done + n, ci + 1\n        log(f"  sim chunk {ci}/{n_chunks}: {done:,}/{len(req):,} rows | {rate:.1f}/s | ok {len(sim):,}")\n    attempted = len(req) - capped\n    ok_share = len(sim) / (attempted - failed) if attempted > failed else float("nan")\n    return sim, dict(n_requests=len(req), n_predicted=len(sim), sim_ok_share=ok_share, sim_errors=errs,\n                     failed_rows=failed, capped_rows=capped, rate=rate, startup_s=startup)\n', 'casmi_scratch/spectra.py': '"""Spectrum cleaning and similarity with the config\'s single max_peaks."""\nimport numpy as np\n\nfrom .kernels import clean, entropy_sim\n\nEMPTY = (np.empty(0, np.float32), np.empty(0, np.float32))\n\n\ndef clean_spectrum(mz, it, cfg):\n    return clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\n                 cfg.int_floor, cfg.max_peaks, cfg.int_power, cfg.ent_weight)\n\n\ndef prep(mz, it, prec, cfg):\n    """Clean a spectrum after dropping peaks above precursor + prec_margin (the organisers\' cleaning)."""\n    mz = np.asarray(mz, np.float32)\n    it = np.asarray(it, np.float32)\n    keep = mz <= prec + cfg.prec_margin\n    return clean(mz[keep], it[keep], cfg.int_floor, cfg.max_peaks, cfg.int_power, cfg.ent_weight)\n\n\ndef esim(a, b, tol):\n    if len(a[0]) == 0 or len(b[0]) == 0:\n        return 0.0\n    return float(entropy_sim(a[0], a[1], b[0], b[1], tol))\n\n\ndef sum_merge(specs, decimals=4):\n    """Sum per-CE spectra (base peak 1 each) the way a merged acquisition does; renormalise to base 1."""\n    parts = [(np.asarray(m, np.float64), np.asarray(i, np.float64)) for m, i in specs if len(m)]\n    if not parts:\n        return EMPTY\n    mz = np.concatenate([m for m, _ in parts])\n    it = np.concatenate([i / i.max() for _, i in parts])\n    _, inv = np.unique(np.round(mz, decimals), return_inverse=True)\n    s = np.bincount(inv, weights=it)\n    m = np.bincount(inv, weights=mz) / np.bincount(inv)\n    o = np.argsort(m)\n    return m[o].astype(np.float32), (s[o] / s.max()).astype(np.float32)\n\n\ndef spectrum_sv(q, sims, cfg):\n    """Simulated-library score of one prepped query spectrum against one candidate.\n\n    sims: the candidate\'s raw predictions (mz, it, prec) at each of the spectrum\'s CEs, None where not simulated.\n    max / mid: best similarity over the CEs; sum: similarity to the renormalised sum. Nothing available -> 0.\n    """\n    have = [s for s in sims if s is not None and len(s[0])]\n    if not have or len(q[0]) == 0:\n        return 0.0\n    if cfg.merge_rule == "sum":\n        mz, it = sum_merge([(m, i) for m, i, _ in have])\n        return esim(q, prep(mz, it, have[0][2], cfg), cfg.mz_tol)\n    return max(esim(q, prep(m, i, p, cfg), cfg.mz_tol) for m, i, p in have)\n\n\nMERGE_RULES = ("max", "sum", "mid")\n\n\ndef ce_tuple(v):\n    """collision_energy_ev cell (list / array / None) -> tuple of floats rounded to 0.1 eV."""\n    if v is None:\n        return ()\n    try:\n        return tuple(round(float(x), 1) for x in v)\n    except TypeError:\n        return ()\n\n\ndef spectrum_ces(ces, rule, default):\n    """One spectrum\'s CE list -> (CEs to simulate, defaulted). CE <= 0 dropped; `mid` simulates the mean once."""\n    assert rule in MERGE_RULES, rule\n    v = [round(float(c), 1) for c in ces if np.isfinite(c) and c > 0]\n    if not v:\n        return [float(default)], True\n    if rule == "mid":\n        return [round(float(np.mean(v)), 1)], False\n    return sorted(set(v)), False\n', 'casmi_scratch/submit.py': '"""Run-mode detection and submission writing."""\nimport os\n\nimport pandas as pd\n\n\ndef run_mode(n_mol, n_spec, cfg, env=None):\n    """-> (fast, info). Fail-safe: full run unless the PUBLIC test file is recognised and this is not the rerun."""\n    env = os.environ if env is None else env\n    raw = env.get("KAGGLE_IS_COMPETITION_RERUN")\n    looks_public = n_mol == cfg.public_n_mol and n_spec == cfg.public_n_spec\n    fast = bool(cfg.allow_fast) and looks_public and not bool(raw) and not cfg.force_full\n    return fast, dict(rerun_env=raw, looks_public=looks_public, mode="fast" if fast else "full")\n\n\ndef make_submission(ranked, sample, topn=25, pad="CCO"):\n    """{molecule_id: [smiles]} aligned to sample_submission, exactly topn per row, audited."""\n    sub = sample[["molecule_id"]].copy()\n    sub["smiles"] = [";".join((list(ranked.get(m, [])) + [pad] * topn)[:topn]) for m in sub.molecule_id]\n    assert len(sub) == len(sample) and (sub.molecule_id.values == sample.molecule_id.values).all()\n    assert not sub.molecule_id.duplicated().any() and sub.smiles.notna().all()\n    assert (sub.smiles.str.split(";").map(len) == topn).all()\n    return sub\n\n\ndef write_submission(sub, path="submission.csv"):\n    sub.to_csv(path, index=False)\n    return path\n\n\ndef molecules(test_df):\n    return [(m, g) for m, g in test_df.groupby("molecule_id")]\n\n\ndef read_sample(path):\n    return pd.read_csv(path)\n'}
PKG_ROOT = WORK / "pkg"
for rel, src in PKG_FILES.items():
    p = PKG_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(src, encoding="utf-8")
sys.path.insert(0, str(PKG_ROOT))


def find_file(name):
    hits = glob.glob(f"{INPUT}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found in {INPUT} -- is the dataset attached?")
    return sorted(hits, key=len)[0]


whl = glob.glob(f"{INPUT}/**/rdkit-*.whl", recursive=True)
if whl:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", whl[0]], check=False)
import rdkit  # noqa: E402  (fails loudly: the pool and MetFrag-lite need RDKit)
print(f"[INFO] RDKit {rdkit.__version__}")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import pyarrow.parquet as pq  # noqa: E402

import casmi_scratch  # noqa: E402
from casmi_scratch import pipeline, submit  # noqa: E402
from casmi_scratch.config import Config  # noqa: E402
from casmi_scratch.library import load_library  # noqa: E402
from casmi_scratch.pool import build_r0_pool  # noqa: E402
from casmi_scratch.sim import setup_iceberg, smoke  # noqa: E402

CFG = Config().with_(**CONFIG_OVERRIDES)
COMP = os.path.dirname(find_file("test.parquet"))
TIMES = {}
print(f"[INFO] casmi_scratch {casmi_scratch.__version__} | {RUN_TAG} | {CFG.as_dict()}")

In [ ]:
t = time.time()
WORKER = setup_iceberg(INPUT, WORK / "sim", CFG, venv=str(WORK / "iceberg_venv")) if CFG.use_sim else None
SMOKE = smoke(WORKER, CFG) if WORKER else None
TIMES["iceberg_setup_s"] = round(time.time() - t, 1)
print(f"[ICEBERG] {SMOKE} in {TIMES['iceberg_setup_s']}s")

In [ ]:
t = time.time()
POOL = build_r0_pool(os.path.dirname(find_file("coco_meta.pkl")), os.path.join(COMP, "train.parquet"),
                     workers=CFG.frag_workers)
TIMES["pool_s"] = round(time.time() - t, 1)
t = time.time()
LIB = load_library(os.path.join(COMP, "train.parquet"), struct_mass=CFG.struct_mass)
TIMES["library_s"] = round(time.time() - t, 1)

TE = pq.read_table(os.path.join(COMP, "test.parquet")).to_pandas()
SAMPLE = pd.read_csv(os.path.join(COMP, "sample_submission.csv"))
MOLS = submit.molecules(TE)
FAST, MODE = submit.run_mode(len(MOLS), len(TE), CFG)
DEEP = {m for m, _ in MOLS[:CFG.fast_dev_molecules]} if FAST else None
print("=" * 78)
print(f"  {MODE} | {len(TE):,} spectra | {len(MOLS):,} molecules")
print("  *** FAST COMMIT: NOT A REAL SUBMISSION ***" if FAST else "  FULL RUN: submittable")
print("=" * 78)

In [ ]:
RANKED, REPORT, AUX = pipeline.run(MOLS, LIB, POOL, CFG, call=WORKER, deep=DEEP, t0=T0, strict=FAST)

In [ ]:
SUB = submit.make_submission(RANKED, SAMPLE, CFG.topn)
submit.write_submission(SUB, WORK / "submission.csv")
REPORT.update(run_tag=RUN_TAG, mode=MODE["mode"], run_mode=MODE, config=CFG.as_dict(), smoke=SMOKE, setup_times=TIMES,
              total_s=round(time.time() - T0, 1), pool_size=len(POOL), library_size=len(LIB))
(WORK / "run_report.json").write_text(json.dumps(REPORT, indent=1, default=str))
print(json.dumps({k: REPORT[k] for k in ("mode", "n_deep", "times", "total_s", "n_requests", "sim_ok_share",
                                          "capped_rows", "failed_rows", "tier1_rate", "tier2_plumbing")},
                 indent=1, default=str))
checks = {
    "sim ok share >= 0.99": (REPORT["sim_ok_share"] >= CFG.min_ok_fast) if CFG.use_sim else None,
    "no capped / failed rows": REPORT["capped_rows"] == 0 and REPORT["failed_rows"] == 0,
    "tier-1 rate >= 0.9 (public file only)": (REPORT["tier1_rate"] >= 0.9) if FAST else None,
}
for k, v in checks.items():
    print(f"  {'PASS ' if v else ('n/a  ' if v is None else 'CHECK')} {k}")
if FAST and CFG.use_sim:
    assert REPORT["sim_ok_share"] >= CFG.min_ok_fast, REPORT["sim_errors"]
SUB.head()